# Lesson 1: Router Engine

Welcome to Lesson 1.

To access the `requirements.txt` file, the data/pdf file required for this lesson and the `helper` and `utils` modules, please go to the `File` menu and select`Open...`.

I hope you enjoy this course!

## Setup

This notebook uses `qwen3.6:27b` through a local Ollama server. If Ollama is not already running as a service, start it in a separate terminal:

```bash
ollama serve
```

Then install and test the model from another terminal:

```bash
ollama pull qwen3.6:27b
ollama run qwen3.6:27b        # Enter /bye after the model responds
```

In [ ]:
import httpx

OLLAMA_URL = "http://localhost:11434"
OLLAMA_MODEL = "qwen3.6:27b"

try:
    ollama_response = httpx.get(f"{OLLAMA_URL}/api/tags", timeout=5)
    ollama_response.raise_for_status()
except (httpx.HTTPError, OSError) as exc:
    raise RuntimeError(
        "Ollama is unavailable. Start it with `ollama serve`."
    ) from exc

installed_models = {
    model["name"] for model in ollama_response.json().get("models", [])
}
if OLLAMA_MODEL not in installed_models:
    raise RuntimeError(
        f"{OLLAMA_MODEL} is not installed. Run `ollama pull {OLLAMA_MODEL}`."
    )

print(f"Ollama is ready with {OLLAMA_MODEL}.")

In [ ]:
# import nest_asyncio

# nest_asyncio.apply()

## Load Data

To download this paper, below is the needed code:

#!wget "https://openreview.net/pdf?id=VtmBAGCN7o" -O metagpt.pdf

**Note**: The pdf file is included with this lesson. To access it, go to the `File` menu and select`Open...`.

In [ ]:
from llama_index.core import SimpleDirectoryReader

# Load documents
DOC_NAME = "5488_MetaGPT_Meta_Programming_.pdf"
documents = SimpleDirectoryReader(
    input_files=[DOC_NAME],
    raise_on_error=True,
).load_data()

## Define LLM and Embedding model

In [ ]:
from llama_index.core.node_parser import SentenceSplitter

splitter = SentenceSplitter(chunk_size=1024)
nodes = splitter.get_nodes_from_documents(documents)

if not documents or not nodes:
    raise ValueError("No document content was extracted.")
if any(
    document.get_content().lstrip().startswith("%PDF-")
    for document in documents
):
    raise RuntimeError(
        "The PDF was read as raw bytes. Install llama-index-readers-file "
        "and restart the kernel."
    )

print("documents:", len(documents))
print("nodes:", len(nodes))
print("characters:", sum(len(node.get_content()) for node in nodes))

In [ ]:
from llama_index.core import Settings
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.llms.openai_like import OpenAILike

Settings.embed_model = HuggingFaceEmbedding(
    model_name="BAAI/bge-small-en-v1.5",
    device="cpu",  # Reserve RTX 4090 VRAM for the generation model.
)

# Settings.llm = OpenAILike(
#     model="google/gemma-4-26b-a4b-it:free",
#     api_key=OPENROUTER_API_KEY,
#     api_base="https://openrouter.ai/api/v1",
#     context_window=262_144,
#     is_chat_model=True,
#     is_function_calling_model=True,
#     temperature=0.1,  # Keep RAG answers grounded and reproducible.
#     max_tokens=512,
#     # This lesson needs direct selection and synthesis, not a reasoning trace.
#     additional_kwargs={
#         "extra_body": {"reasoning": {"enabled": False}},
#     },
# )

Settings.llm = OpenAILike(
    model=OLLAMA_MODEL,
    api_key="ollama",
    api_base=f"{OLLAMA_URL}/v1",
    context_window=98_304,
    is_chat_model=True,
    is_function_calling_model=True,
    reasoning_effort="none",
    temperature=0.1,
    max_tokens=4096,
    max_retries=0,
    timeout=300,
)

# Assign each global setting only once. The original tutorial's OpenAI
# assignments would overwrite the OpenRouter LLM and local embedding model.

## Define Summary Index and Vector Index over the Same Data

In [ ]:
from llama_index.core import SummaryIndex, VectorStoreIndex

summary_index = SummaryIndex(nodes)
vector_index = VectorStoreIndex(nodes)

## Define Query Engines and Set Metadata

In [ ]:
summary_query_engine = summary_index.as_query_engine(
    response_mode="tree_summarize",
    use_async=False,  # Avoid competing generations on one local GPU.
)
vector_query_engine = vector_index.as_query_engine(similarity_top_k=5)

In [ ]:
from llama_index.core.tools import QueryEngineTool

summary_tool = QueryEngineTool.from_defaults(
    query_engine=summary_query_engine,
    description=(
        "Useful for summarization questions related to MetaGPT"
    ),
)

vector_tool = QueryEngineTool.from_defaults(
    query_engine=vector_query_engine,
    description=(
        "Useful for retrieving specific context from the MetaGPT paper."
    ),
)

## Define Router Query Engine

In [ ]:
from llama_index.core.base.base_selector import BaseSelector
from llama_index.core.query_engine.router_query_engine import RouterQueryEngine
from llama_index.core.selectors import LLMSingleSelector, PydanticSingleSelector

selector: BaseSelector
if Settings.llm.metadata.is_function_calling_model:
    selector = PydanticSingleSelector.from_defaults(llm=Settings.llm)
else:
    selector = LLMSingleSelector.from_defaults(llm=Settings.llm)

query_engine = RouterQueryEngine(
    selector=selector,
    query_engine_tools=[
        summary_tool,
        vector_tool,
    ],
    verbose=True,
)

In [ ]:
response = await query_engine.aquery("What is the summary of the document?")
print(str(response))

In [ ]:
print(len(response.source_nodes))

In [ ]:
response = query_engine.query(
    "How do agents share information with other agents?"
)
print(str(response))

## Let's put everything together

In [ ]:
from utils import get_router_query_engine

query_engine = get_router_query_engine(DOC_NAME)

In [ ]:
response = query_engine.query("Tell me about the ablation study results?")
print(str(response))